# Phase 3 - Activation Probing and Representation Extraction

Esta fase observa representacoes internas do modelo. O objetivo nao e aplicar steering ainda, mas medir se respostas corretas e incorretas se separam em alguma camada.

## Objetivo

Usamos JSONL da Fase 2, especialmente tarefas `sampling_sensitive` e `fragile`, para extrair vetores por camada e procurar uma direcao latente candidata.

Esta celula prepara paths e imports basicos.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


Imports analiticos. Tudo pesado fica nos modulos do projeto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from slm_steering.activations.storage import load_activation_dataset
from slm_steering.activations.probing import centroid_separability_frame
from slm_steering.activations.probing import probe_layers, latent_directions
from slm_steering.activations.probing import pca_2d, best_layer_candidate


Defina aqui o diretorio criado por `scripts/extract_activations.py`.

In [ ]:
ACTIVATION_DIR = ROOT / "runs" / "phase3" / "qwen_phase3_probe"
ACTIVATION_DIR.exists()


Comando de exemplo. Ajuste `--input-jsonl` e `--summary-json` para uma run da Fase 2.

In [ ]:
print("python scripts/extract_activations.py --input-jsonl runs/phase2/qwen15b_temp08_n5.jsonl --summary-json runs/phase2/qwen15b_temp08_n5_summary.json --output-dir runs/phase3/qwen_phase3_probe --layers 0:28:4 --token-position completion_last --difficulty-filter sampling_sensitive,fragile --local-files-only")


Carregamos o dataset quando ele existe. Se ainda nao existir, as proximas celulas permanecem vazias.

In [ ]:
dataset = load_activation_dataset(ACTIVATION_DIR) if ACTIVATION_DIR.exists() else None
dataset.num_samples if dataset else 0


Quantidade de amostras corretas e incorretas. Sem ambas as classes, nao ha separabilidade a medir.

In [ ]:
if dataset:
    counts = pd.Series(dataset.labels).map({0: "incorrect", 1: "correct"}).value_counts()
else:
    counts = pd.Series(dtype=int)
counts


Metadados por tentativa: tarefa, tentativa, label, dificuldade e posicao do token extraido.

In [ ]:
dataset.metadata.head() if dataset else pd.DataFrame()


Separabilidade por camada usando distancia entre centroides e razao tipo Fisher.

In [ ]:
separability = centroid_separability_frame(dataset) if dataset else pd.DataFrame()
separability


Visualizamos a distancia entre centroides por camada.

In [ ]:
if not separability.empty:
    separability.plot(x="layer", y="centroid_distance", marker="o")
    plt.title("Distancia entre centroides por camada")
    plt.ylabel("||mean_correct - mean_incorrect||")
    plt.show()


Treinamos probes lineares simples. Eles sao diagnosticos, nao parte do sistema final.

In [ ]:
probes = probe_layers(dataset, steps=300) if dataset else pd.DataFrame()
probes


A melhor camada candidata combina separabilidade de centroides e desempenho do probe.

In [ ]:
best_layer = best_layer_candidate(separability, probes) if dataset else None
best_layer


PCA 2D na melhor camada. Pontos verdes sao corretos; vermelhos sao incorretos.

In [ ]:
if dataset and best_layer in dataset.layers:
    pos = dataset.layers.index(best_layer)
    coords = pca_2d(dataset.activations[:, pos, :])
    colors = np.where(dataset.labels == 1, "#4c9f70", "#d95f5f")
    plt.scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.8)
    plt.title(f"PCA 2D - camada {best_layer}")
    plt.show()


A direcao latente candidata e `mean_correct - mean_incorrect` na melhor camada.

In [ ]:
directions = latent_directions(dataset) if dataset else {}
candidate_direction = directions.get(best_layer)
None if candidate_direction is None else candidate_direction.shape


Norma da direcao candidata. Ela ainda nao deve ser aplicada; por enquanto serve apenas como hipotese para a Fase 4.

In [ ]:
float(np.linalg.norm(candidate_direction)) if candidate_direction is not None else None


## Leitura para apresentacao

- Se centroides se separam em camadas intermediarias/finais, ha sinal representacional associado a sucesso.
- Se probes lineares generalizam, a separacao nao e apenas artefato visual.
- A melhor camada candidata vira alvo para steering futuro.
- Esta fase nao altera geracao; ela apenas observa e mede.